# GitHub Submission — Step by Step

Run each cell **in order**. Every cell is safe to re-run if something goes wrong.

> **Before you start:** Open this notebook in JupyterLab from the folder that contains all your project files.

## Cell 1 — Configure your info

✏️ **Edit the three variables below before running anything else.**

In [ ]:
# ── EDIT THESE THREE LINES ────────────────────────────────────────────────
GITHUB_USERNAME  = "your_github_username"      # e.g. "othman123"
GITHUB_EMAIL     = "your_email@example.com"    # the email linked to your GitHub account
REPO_NAME        = "music-knowledge-graph"     # what the GitHub repo will be called
# ─────────────────────────────────────────────────────────────────────────

print(f"Username : {GITHUB_USERNAME}")
print(f"Email    : {GITHUB_EMAIL}")
print(f"Repo name: {REPO_NAME}")
print("\nLooks good? Proceed to Cell 2.")

## Cell 2 — Install GitHub CLI (gh)

This lets you create the repo and the release directly from the terminal without leaving JupyterLab.

In [ ]:
import subprocess, sys

# Check if gh is already installed
result = subprocess.run(["gh", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print("GitHub CLI already installed:", result.stdout.splitlines()[0])
else:
    print("Installing GitHub CLI via winget...")
    subprocess.run(["winget", "install", "--id", "GitHub.cli", "-e", "--silent"], check=True)
    print("Done. You may need to RESTART JupyterLab once for gh to be on PATH.")
    print("After restarting, re-run from Cell 3.")

## Cell 3 — Authenticate with GitHub

This opens a browser window where you log in once. After that gh remembers you.

In [ ]:
import subprocess

# Check if already authenticated
check = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True)
if "Logged in" in check.stdout or "Logged in" in check.stderr:
    print("Already authenticated with GitHub.")
    print(check.stderr.strip())
else:
    print("Opening browser for GitHub login...")
    print("Select: GitHub.com → HTTPS → Login with a web browser")
    subprocess.run(["gh", "auth", "login"], check=True)
    print("Authentication complete.")

## Cell 4 — Organise files into the required folder structure

This reads all your files from the current folder and sorts them into the
standard project structure required by the grading guide.

In [1]:
import os, shutil
from pathlib import Path

# Root of the project = folder where this notebook lives
ROOT = Path(os.getcwd())
print(f"Project root: {ROOT}")

# ── Folder layout ──────────────────────────────────────────────────────────
FOLDERS = [
    "notebooks",
    "kg_artifacts",
    "data/samples",
    "reports",
    "src",
]
for f in FOLDERS:
    (ROOT / f).mkdir(parents=True, exist_ok=True)
print("Folders created.")

# ── Rules: (glob_pattern, destination_subfolder) ───────────────────────────
MOVE_RULES = [
    # Notebooks
    ("*.ipynb",                 "notebooks"),
    # KG artifacts
    ("*.ttl",                   "kg_artifacts"),
    ("*.nt",                    "kg_artifacts"),
    ("alignment*.csv",          "kg_artifacts"),
    ("entity_mapping*.csv",     "kg_artifacts"),
    ("predicate_alignment*.csv","kg_artifacts"),
    # KGE train/valid/test splits
    ("train.txt",               "data"),
    ("valid.txt",               "data"),
    ("test.txt",                "data"),
    ("*.tsv",                   "data"),
    # Reports and visuals
    ("*.pdf",                   "reports"),
    ("*.docx",                  "reports"),
    ("*.png",                   "reports"),
    # Evaluation CSVs
    ("kge_results.csv",         "reports"),
    ("rag_evaluation.csv",      "reports"),
    ("seed_artists.csv",        "data/samples"),
    ("seed_albums.csv",         "data/samples"),
    ("ner_entities.csv",        "data/samples"),
]

moved = []
for pattern, dest in MOVE_RULES:
    for src in ROOT.glob(pattern):
        # Don't move files that are already inside a subfolder
        if src.parent != ROOT:
            continue
        # Don't move this notebook itself to notebooks/ yet (do it last)
        if src.name == "github_submission.ipynb":
            continue
        dst = ROOT / dest / src.name
        if not dst.exists():
            shutil.move(str(src), str(dst))
            moved.append(f"  {src.name}  →  {dest}/")

if moved:
    print("Moved:")
    for m in moved:
        print(m)
else:
    print("No files to move (already organised or nothing matched).")

# Move this notebook to notebooks/ last
this_nb = ROOT / "github_submission.ipynb"
if this_nb.exists():
    shutil.move(str(this_nb), str(ROOT / "notebooks" / "github_submission.ipynb"))
    print("  github_submission.ipynb  →  notebooks/")

print("\nFinal structure:")
for item in sorted(ROOT.rglob("*")):
    if ".git" in item.parts:
        continue
    depth = len(item.relative_to(ROOT).parts) - 1
    print("  " + "  " * depth + item.name)

Project root: C:\Users\othma\Desktop\EsilvDocs\Datamining\Lab4
Folders created.
Moved:
  Lab4.ipynb  →  notebooks/
  step0_crawler_ner.ipynb  →  notebooks/
  step1_build_knowledge_base.ipynb  →  notebooks/
  step2_entity_linking.ipynb  →  notebooks/
  step3_predicate_alignment.ipynb  →  notebooks/
  step4_kb_expansion.ipynb  →  notebooks/
  step5_swrl_reasoning.ipynb  →  notebooks/
  step6_kge_pykeen.ipynb  →  notebooks/
  step7_rag_ollama.ipynb  →  notebooks/
  music_kb.ttl  →  kg_artifacts/
  music_kb_aligned.ttl  →  kg_artifacts/
  music_kb_linked.ttl  →  kg_artifacts/
  music_kg_kge.ttl  →  kg_artifacts/
  entity_mapping_table.csv  →  kg_artifacts/
  predicate_alignment_table.csv  →  kg_artifacts/
  train.txt  →  data/
  valid.txt  →  data/
  test.txt  →  data/
  music_kg_entities.tsv  →  data/
  music_kg_relations.tsv  →  data/
  music_kg_triples.tsv  →  data/
  music_kg_final_report.docx  →  reports/
  size_sensitivity.png  →  reports/
  tsne_embeddings.png  →  reports/
  kge_res

## Cell 5 — Generate requirements.txt and .gitignore

In [2]:
from pathlib import Path
import os

ROOT = Path(os.getcwd())

# ── requirements.txt ──────────────────────────────────────────────────────
reqs = """# Python dependencies for the Music Knowledge Graph project
# Install with: pip install -r requirements.txt

# Core RDF / SPARQL
rdflib>=6.3.2
SPARQLWrapper>=2.0.0
owlready2>=0.46

# Data collection
requests>=2.31.0
pandas>=2.0.0
tqdm>=4.65.0

# NER
spacy>=3.7.0
# After install run: python -m spacy download en_core_web_sm

# Knowledge Graph Embeddings
pykeen>=1.10.0
torch>=2.0.0

# Visualisation
matplotlib>=3.7.0
scikit-learn>=1.3.0

# RAG
# Ollama must be installed separately: https://ollama.ai
# Then: ollama pull llama3
"""

(ROOT / "requirements.txt").write_text(reqs)
print("requirements.txt written.")

# ── .gitignore ─────────────────────────────────────────────────────────────
gitignore = """# Python
__pycache__/
*.py[cod]
*.pyo
.env
.venv/
env/
venv/

# Jupyter
.ipynb_checkpoints/
*.ipynb_checkpoints

# Large data files — keep samples, ignore full dumps
data/music_kg_triples.tsv
data/music_kg_entities.tsv
data/music_kg_kge.ttl
kg_artifacts/expanded.nt

# OS
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/

# Logs
*.log
"""

(ROOT / ".gitignore").write_text(gitignore)
print(".gitignore written.")

# ── data/README.md ─────────────────────────────────────────────────────────
data_readme = """# Data

## samples/
Small representative samples of the crawled data:
- `seed_artists.csv` — 20 seed artists from MusicBrainz
- `seed_albums.csv` — 160 albums
- `ner_entities.csv` — 86 NER-extracted entities

## Full dataset
The full expanded KB (51,806 triples) is too large for GitHub.
Generate it by running the notebooks in order:
1. `notebooks/step0_crawler_ner.ipynb`
2. `notebooks/step1_build_knowledge_base.ipynb`
3. `notebooks/step2_entity_linking.ipynb`
4. `notebooks/step3_predicate_alignment.ipynb`
5. `notebooks/step4_kb_expansion.ipynb`

This will produce `music_kg_triples.tsv`, `music_kg_kge.ttl`, etc.
"""

(ROOT / "data" / "README.md").write_text(data_readme)
print("data/README.md written.")

requirements.txt written.
.gitignore written.
data/README.md written.


## Cell 6 — Generate README.md

This writes the full professional README to the project root.

In [3]:
import os
from pathlib import Path

ROOT = Path(os.getcwd())

README = f"""# Music Knowledge Graph

> **Knowledge Engineering & Semantic Web — Final Project**

A complete end-to-end semantic web pipeline over the Music domain:
data collection → RDF knowledge base → Wikidata alignment → SPARQL expansion →
SWRL reasoning → knowledge graph embeddings → RAG question answering.

---

## Pipeline Overview

```
MusicBrainz API
      │
      ▼
[Step 0] Crawl + Clean + NER          → seed_artists.csv, seed_albums.csv
      │
      ▼
[Step 1] Build RDF Knowledge Base     → music_kb.ttl          (498 triples)
      │
      ▼
[Step 2] Entity Linking (Wikidata)    → music_kb_linked.ttl   (848 triples)
      │
      ▼
[Step 3] Predicate Alignment          → music_kb_aligned.ttl  (876 triples)
      │
      ▼
[Step 4] SPARQL Expansion             → music_kg_kge.ttl      (51,806 triples)
      │
      ▼
[Step 5] SWRL Reasoning (OWLReady2)   → inferred class memberships
      │
      ▼
[Step 6] KGE — TransE + RotatE        → train.txt / valid.txt / test.txt
      │
      ▼
[Step 7] RAG — Ollama llama3          → NL → SPARQL → answer
```

---

## Results at a Glance

| Metric | Value |
|--------|-------|
| Seed artists crawled | 20 (160 albums) |
| Final KB size | 51,806 triples · 42,032 entities · 439 relations |
| Entity alignments (owl:sameAs) | 67 (64 Wikidata + 3 DBpedia) |
| Predicate alignments | 14 (6 equivalent · 5 subProperty · 3 related) |
| TransE Hits@1 / Hits@10 | 0.097% / 1.94% |
| RotatE Hits@1 / Hits@10 | **1.93% / 3.45%** (20× better at rank 1) |
| RotatE Harmonic Mean Rank | 40.4 (vs 122.8 for TransE) |
| Size sensitivity Hits@10 | 20K: 3.7% → 50K: 18.4% → Full: **21.7%** |
| RAG success rate | 1/5 with schema injection vs 0/5 baseline |

---

## Repository Structure

```
music-knowledge-graph/
├── notebooks/                  # All Jupyter notebooks (run in order)
│   ├── step0_crawler_ner.ipynb
│   ├── step1_build_knowledge_base.ipynb
│   ├── step2_entity_linking.ipynb
│   ├── step3_predicate_alignment.ipynb
│   ├── step4_kb_expansion.ipynb
│   ├── step5_swrl_reasoning.ipynb
│   ├── step6_kge_pykeen.ipynb
│   └── step7_rag_ollama.ipynb
├── kg_artifacts/               # RDF files
│   ├── music_kb.ttl            # Seed KB (498 triples)
│   ├── music_kb_linked.ttl     # After entity linking
│   ├── music_kb_aligned.ttl    # After predicate alignment
│   ├── entity_mapping_table.csv
│   └── predicate_alignment_table.csv
├── data/
│   ├── samples/                # Small CSV samples (included)
│   │   ├── seed_artists.csv
│   │   ├── seed_albums.csv
│   │   └── ner_entities.csv
│   ├── train.txt               # KGE training split
│   ├── valid.txt               # KGE validation split
│   └── test.txt                # KGE test split
├── reports/
│   ├── final_report.pdf
│   ├── kge_results.csv
│   ├── rag_evaluation.csv
│   ├── size_sensitivity.png
│   └── tsne_embeddings.png
├── requirements.txt
├── .gitignore
└── README.md
```

---

## Installation

### 1. Clone the repository
```bash
git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
cd {REPO_NAME}
```

### 2. Create and activate a conda environment
```bash
conda create -n musickg python=3.11
conda activate musickg
```

### 3. Install Python dependencies
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

### 4. Install Ollama (for RAG — Step 7 only)
Download from [https://ollama.ai](https://ollama.ai), then:
```bash
ollama pull llama3
```

---

## How to Run

Open JupyterLab from the project root and run notebooks **in order**:

```bash
jupyter lab
```

| Notebook | What it does | Runtime |
|----------|-------------|--------|
| `step0_crawler_ner.ipynb` | Crawl MusicBrainz, clean, run NER | ~5 min (API rate limit) |
| `step1_build_knowledge_base.ipynb` | Build seed RDF KB | <1 min |
| `step2_entity_linking.ipynb` | Align entities to Wikidata/DBpedia | ~3 min |
| `step3_predicate_alignment.ipynb` | Align predicates via SPARQL | ~5 min |
| `step4_kb_expansion.ipynb` | Expand KB to 50K+ triples | ~30 min (SPARQL) |
| `step5_swrl_reasoning.ipynb` | SWRL rules — no Java needed | <1 min |
| `step6_kge_pykeen.ipynb` | Train TransE + RotatE | ~20 min (GPU) / ~60 min (CPU) |
| `step7_rag_ollama.ipynb` | RAG demo — requires Ollama running | interactive |

### Hardware requirements
- RAM: 8 GB minimum, 16 GB recommended (for Step 4 expansion)
- GPU: optional but strongly recommended for Step 6 (CUDA)
- Disk: ~500 MB for full expanded KB

### Running the RAG demo
1. Start Ollama in a separate terminal: `ollama serve`
2. Run all cells in `step7_rag_ollama.ipynb`
3. The last cell opens an interactive CLI — type any music question

Example questions:
```
Which record label is Adele signed to?
What genre does Kendrick Lamar play?
Which artists collaborated with Beyonce?
```

---

## Key Design Decisions

**RDF Modelling:** Two-namespace design (`mus:` for entities, `prop:` for predicates).
All literals are typed with XSD datatypes. `prop:collaboratedWith` is declared
`owl:SymmetricProperty`; `prop:releasedAlbum` and `prop:producedBy` are `owl:inverseOf`.

**Entity Linking:** Three-factor confidence score (label similarity 50%,
type keyword match 30%, rank position 20%). Threshold 0.55.
All 67 seed entities linked at ≥ 0.97 confidence.

**Predicate Alignment:** Two SPARQL strategies:
- Label keyword search: `FILTER(CONTAINS(LCASE(?propLabel), "keyword"))`
- Instance validation: `wd:Subject ?prop wd:Object`

**SWRL Reasoning:** Pellet-free Python forward-chaining (no Java dependency).
Produces semantically equivalent results to a full SWRL reasoner.

**KGE:** RotatE outperforms TransE on Hits@1 (1.93% vs 0.097%) because the KB
contains symmetric (`collaboratedWith`) and inverse (`releasedAlbum`/`producedBy`)
relation patterns that RotatE is specifically designed to handle.

**RAG:** Schema injection (auto-generated from RDF graph) is the critical component.
Without it the LLM generates wrong namespace prefixes and hallucinated property URIs.

---

## Technologies

| Component | Technology |
|-----------|------------|
| RDF / SPARQL | RDFLib 6, SPARQLWrapper |
| Open KB alignment | Wikidata API, DBpedia Spotlight |
| Ontology / Reasoning | OWLReady2 |
| Knowledge Graph Embeddings | PyKEEN 1.10, PyTorch 2.x |
| Embedding visualisation | scikit-learn t-SNE, matplotlib |
| LLM (RAG) | Ollama llama3 (local) |
| NER | spaCy en_core_web_sm |
| Data source | MusicBrainz API (CC0) |

---

## License

Code: MIT License  
Data: MusicBrainz data is CC0. Wikidata data is CC0.
"""

(ROOT / "README.md").write_text(README)
print("README.md written — preview:")
print()
with open(ROOT / "README.md") as f:
    lines = f.readlines()
print(f"Total lines: {len(lines)}")
print("".join(lines[:30]))

NameError: name 'GITHUB_USERNAME' is not defined

## Cell 7 — Configure Git and initialise the repo

This sets your identity and creates the local git repo.

In [ ]:
import subprocess, os
from pathlib import Path

ROOT = Path(os.getcwd())
os.chdir(ROOT)

def run(cmd, **kwargs):
    """Run a shell command and print output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:", result.stderr.strip())
    return result

# Configure git identity
run(f'git config --global user.name "{GITHUB_USERNAME}"')
run(f'git config --global user.email "{GITHUB_EMAIL}"')
run('git config --global init.defaultBranch main')

# Init repo if not already done
if not (ROOT / ".git").exists():
    run('git init')
    print("Git repo initialised.")
else:
    print("Git repo already exists.")

print("\nGit config:")
run('git config --list --local')

## Cell 8 — Create the GitHub repo online (using gh CLI)

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print("STDERR:", result.stderr.strip())
    return result

# Create the remote repo on GitHub (public)
print(f"Creating GitHub repo: {GITHUB_USERNAME}/{REPO_NAME}")
result = run(f'gh repo create {REPO_NAME} --public --description "Music Knowledge Graph — KE & Semantic Web final project" --source=. --remote=origin')

if result.returncode == 0:
    print(f"\nRepo created: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")
else:
    # Repo may already exist — just add the remote
    print("Repo may already exist, adding remote manually...")
    run(f'git remote remove origin')
    run(f'git remote add origin https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git')
    print(f"Remote set to: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git")

run('git remote -v')

## Cell 9 — Stage, commit, and push everything

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print("STDERR:", result.stderr.strip())
    return result

# Stage all files
print("Staging files...")
run('git add -A')

# Show what will be committed
print("\nFiles staged:")
run('git status --short')

# Commit
print("\nCommitting...")
run('git commit -m "Initial commit — complete Music Knowledge Graph pipeline"')

# Push to main
print("\nPushing to GitHub...")
result = run('git push -u origin main')

if result.returncode == 0:
    print(f"\nSuccessfully pushed to https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")
else:
    print("Push failed — see STDERR above.")
    print("Common fix: run 'gh auth login' again in a terminal, then re-run this cell.")

## Cell 10 — Create the v1.0-final release tag

This is the release link you submit to DVL.

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print("STDERR:", result.stderr.strip())
    return result

# Create the release via gh CLI
print("Creating release v1.0-final...")
release_notes = """## Music Knowledge Graph — Final Submission

### Pipeline
- Data collection from MusicBrainz API (20 artists, 160 albums)
- RDF KB construction: 498 seed triples → 51,806 after SPARQL expansion
- Entity linking: 67 entities aligned to Wikidata/DBpedia (confidence ≥ 0.97)
- Predicate alignment: 14 predicates (6 equivalent, 5 subPropertyOf, 3 relatedMatch)
- SWRL reasoning: 3 inferred classes, 5 new relation instances
- KGE: TransE + RotatE (RotatE Hits@1 = 1.93%, 20× better than TransE)
- RAG: Ollama llama3, NL→SPARQL with self-repair loop

### Key files
- `notebooks/` — all 8 pipeline notebooks
- `kg_artifacts/` — RDF Turtle files at each stage
- `data/` — KGE train/valid/test splits
- `reports/` — final report PDF, result charts
"""

# Write release notes to a temp file
with open("release_notes.md", "w") as f:
    f.write(release_notes)

result = run('gh release create v1.0-final --title "v1.0-final — Final Submission" --notes-file release_notes.md')

import os
os.remove("release_notes.md")

if result.returncode == 0:
    print(f"\nRelease created!")
    print(f"Release URL: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/releases/tag/v1.0-final")
    print(f"\n{'='*60}")
    print("SUBMIT THIS LINK ON DVL:")
    print(f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/releases/tag/v1.0-final")
    print(f"{'='*60}")
else:
    print("Release creation failed — see error above.")

## Cell 11 — Verify everything looks correct

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip(): print(result.stdout.strip())
    return result

print("=" * 60)
print("FINAL CHECKLIST")
print("=" * 60)

# Check remote
result = subprocess.run('git remote get-url origin', shell=True, capture_output=True, text=True)
print(f"\nRepo URL   : {result.stdout.strip()}")

# Check latest commit
result = subprocess.run('git log --oneline -3', shell=True, capture_output=True, text=True)
print(f"Last commits:\n{result.stdout.strip()}")

# Check release
result = subprocess.run('gh release list', shell=True, capture_output=True, text=True)
print(f"\nReleases:\n{result.stdout.strip()}")

print(f"""
{'='*60}
SUBMIT TO DVL:

  GitHub release link:
  https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/releases/tag/v1.0-final

  Report PDF:
  reports/final_report.pdf
{'='*60}
""")

## Troubleshooting

### Push was rejected (non-fast-forward)
```bash
git pull origin main --rebase
git push origin main
```

### gh: command not found after installing
Close JupyterLab completely, reopen it, then re-run from Cell 3.

### Authentication error on push
Run this in a Git Bash terminal (not in the notebook):
```bash
gh auth login
```
Choose: GitHub.com → HTTPS → Login with a web browser.

### Large file warning (file over 100MB)
Add it to `.gitignore` (Cell 5 already covers the biggest files).
If it was already staged: `git rm --cached data/music_kg_triples.tsv`

### Want to make a change and re-release
```python
# In a new cell:
import subprocess
subprocess.run('git add -A && git commit -m "Fix: updated report" && git push', shell=True)
subprocess.run('gh release delete v1.0-final --yes', shell=True)
subprocess.run('gh release create v1.0-final --title "v1.0-final" --notes "Updated submission"', shell=True)
```